# MS MARCO RARS-v2 boundary-loss feasibility (Colab T4)

Primary relevance-supervised development run on the corpus-aligned 4,980/1,000 MS MARCO clean split. BEIR NQ test and post-hoc artifacts are not read.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, shutil, subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9'], check=True)
DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2')
BRANCH = 'experiment/rars-v2-boundary-consolidation'
CORE_COMMIT = '07aed4aa6b6d07d1ee6e860c0e0632136231b230'
BUNDLES = Path('/content/rars-v2.1-msmarco-work/bundles')
OUTPUT = Path('/content/drive/MyDrive/rars-v2.1-boundary-loss-msmarco')
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', '--detach', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git',
                    str(REPO)], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert subprocess.run(['git', '-C', str(REPO), 'merge-base', '--is-ancestor', CORE_COMMIT, head]).returncode == 0
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
required = [CACHE/'embeddings.fp16.memmap', CACHE/'doc_ids.int64.memmap',
            CACHE/'query_vectors.fp32.npy', CACHE/'qrels_subset.json', INDEX,
            CLEAN/'selected_config.json', PCA/'bases/pca_unweighted_rank16.float32.npy']
missing = [str(path) for path in required if not path.exists()]
assert not missing, missing
assert shutil.disk_usage('/content').free / 1e9 >= 3, 'Need 3 GB local disk'
print('Clean checkout:', head)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
  'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
  'tests/test_boundary_loss_sidecar.py',
  'tests/test_boundary_aware_sidecar.py'], cwd=REPO, check=True)

## Build corpus-aligned compact bundles on local Colab disk

In [ ]:
builder = [
 sys.executable, str(REPO/'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
 '--embeddings', str(CACHE/'embeddings.fp16.memmap'),
 '--doc-ids', str(CACHE/'doc_ids.int64.memmap'),
 '--query-vectors', str(CACHE/'query_vectors.fp32.npy'),
 '--index', str(INDEX), '--qrels', str(CACHE/'qrels_subset.json'),
 '--train-split', str(REPO/'splits/msmarco_rars_train_split.json'),
 '--validation-split', str(REPO/'splits/msmarco_rars_validation_split.json'),
 '--cache-root', str(CLEAN/'candidate_cache'),
 '--pca-config', str(REPO/'results/rars_pca_comparator/selected_pca_config.json'),
 '--pca-basis', str(PCA/'bases/pca_unweighted_rank16.float32.npy'),
 '--pca-scales', str(PCA/'sidecars/scales_pca_rank16.float32.npy'),
 '--pca-codes', str(PCA/'sidecars/codes_pca_rank16.int8.memmap'),
 '--rars-config', str(CLEAN/'selected_config.json'),
 '--rars-basis', str(CLEAN/'bases/score_error_weighted_rank16.npy'),
 '--rars-scales', str(CLEAN/'sidecars/scales_score_error_weighted_rank16.float32.npy'),
 '--rars-codes', str(CLEAN/'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
 '--output-root', str(BUNDLES), '--residual-batch-size', '20000']
cached_residuals = CLEAN/'residual_ivfpq_m32.float32.memmap'
if cached_residuals.exists(): builder += ['--cached-full-residuals', str(cached_residuals)]
subprocess.run(builder, check=True)
bundle_summary = json.loads((BUNDLES/'bundle_build_summary.json').read_text())
print(json.dumps(bundle_summary, indent=2))
subprocess.run(['du', '-sh', str(BUNDLES)], check=True)

## Seed-42 smoke test (1 epoch)

In [ ]:
def train_run(output_dir, epochs):
    subprocess.run([sys.executable, str(REPO/'scripts/train_boundary_loss_sidecar.py'),
      '--bundle-dir', str(BUNDLES/'inner_train'),
      '--selection-bundle-dir', str(BUNDLES/'inner_validation'),
      '--validation-bundle-dir', str(BUNDLES/'outer_validation'),
      '--output-dir', str(output_dir), '--rank', '16', '--top-b', '40',
      '--final-k', '10', '--epochs', str(epochs), '--batch-size', '2048',
      '--learning-rate', '0.0001', '--correction-l2', '0.001',
      '--max-correction', '0.05', '--initial-gate-bias', '-2.0',
      '--seed', '42', '--skip-full-encoding', '--device', 'cuda'], check=True)
    return json.loads((output_dir/'training_summary.json').read_text())
smoke = train_run(OUTPUT/'v2.1-seed42-smoke-1epoch', 1)
assert smoke['test_qrels_accessed'] is False and smoke['final_loss'] == smoke['final_loss']
print(json.dumps(smoke, indent=2))

## Registered v2.1 seed-42 run (maximum 10 epochs) and go/no-go

In [ ]:
run_dir = OUTPUT/'v2.1-seed42-max10epochs'
result = train_run(run_dir, 10)
m = result['validation']; retained = m['int8_fraction_of_fp32_gain']
checks = {
 'gain_over_base_at_least_0.01': m['int8_gain_over_base'] >= 0.01 - 1e-12,
 'beats_storage_matched_pca': m.get('beats_storage_matched_pca', False),
 'retains_70pct_fp32_gain': retained is not None and retained >= 0.70,
 'improved_exceeds_harmed': m['improved_queries'] > m['harmed_queries']}
decision = {'status': 'GO' if all(checks.values()) else 'NO_GO_OR_REVISE',
            'checks': checks, 'validation': m, 'test_qrels_accessed': False}
(run_dir/'go_no_go.json').write_text(json.dumps(decision, indent=2)+'\n')
print(json.dumps(decision, indent=2))